In [1]:
%pip install numpy datasets google.generativeai python_dotenv


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 1: Fixed Dataset Loading
import numpy as np
from datasets import load_dataset

def load_and_sample_stories():
    dataset = load_dataset('euclaise/writingprompts', split='train')
    indices = np.random.choice(len(dataset), 120, replace=False)
    indices = [int(i) for i in indices]  # Convert numpy.int64 to native int
    return [dataset[i]['story'] for i in indices], indices

stories, story_indices = load_and_sample_stories()


In [ ]:
# Cell 2: Sequential API Calls with Rate Limiting and Incremental Saving
import google.generativeai as genai
import os
import re
import time
from dotenv import load_dotenv
from datetime import datetime
import json

load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

def process_story_with_gemini_sequential(story, index):
    model = genai.GenerativeModel('gemini-1.5-flash')
    
    # Call 1: Objective summary
    prompt_summary = f"""Provide an objective summary of the following story. Refer to the narrator as <the narrator>.

Story: {story[:20000]}"""
    response_summary = model.generate_content(prompt_summary)
    summary = response_summary.text.strip()
    print(f"  - Summary completed")
    
    # Call 2: List of characters
    prompt_characters = f"""List all characters in the following story. Include all named entities.

Story: {story[:20000]}"""
    response_characters = model.generate_content(prompt_characters)
    characters = [c.strip() for c in response_characters.text.strip().split('\n') if c.strip()]
    print(f"  - Characters completed")
    
    # Call 3: Short camelCase story ID
    prompt_name = f"""Create a short camelCase name for the following story. This will be used as a story ID.

Story: {story[:20000]}"""
    response_name = model.generate_content(prompt_name)
    name = response_name.text.strip()
    print(f"  - Name completed")
    
    # Call 4: List of environments/settings
    prompt_environment = f"""List all environments and settings in the following story.

Story: {story[:20000]}"""
    response_environment = model.generate_content(prompt_environment)
    environment = [e.strip() for e in response_environment.text.strip().split('\n') if e.strip()]
    print(f"  - Environment completed")
    
    return {
        "storyNum": int(index),
        "name": name,
        "source": story,
        "summary": summary,
        "characters": characters,
        "environment": environment
    }

def save_current_progress(data, filename):
    """Save current progress to file"""
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Progress saved to {filename}")

# Initialize filename for incremental saving
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"extracted_{timestamp}.json"

# Process stories and save results incrementally
extracted_data = []
for idx, (story, story_idx) in enumerate(zip(stories, story_indices)):
    time.sleep(10)  # Rate limiting
    print(f"Processing story {idx+1}/120 (index: {story_idx})")
    try:
        result = process_story_with_gemini_sequential(story, story_idx)
        extracted_data.append(result)
        # Save after each successful processing
        save_current_progress(extracted_data, filename)
        print(f"Completed {idx+1}/120 stories")
    except Exception as e:
        print(f"Error processing story {idx+1} (index: {story_idx}): {str(e)}")
        # Save even if there's an error
        save_current_progress(extracted_data, filename)
        print("Continuing with next story...")
    
    # Additional rate limiting between stories
    if idx < len(stories) - 1:  # Don't sleep after the last story
        print("Waiting 5 seconds before next story...")
        time.sleep(5)

print(f"All processing complete. Final data saved to {filename}")


In [ ]:
# Cell 4: Inference Functions with Ollama/Mistral Implementation
import csv
import time
import os
import requests
import json

def generate_retelling(extracted_data, output_dir="output"):
    # Format the prompt with data from the extracted story
    characters_str = ", ".join(extracted_data["characters"])
    environments_str = ", ".join(extracted_data["environment"])
    
    prompt_template = f"""Here is a list of objective, third person facts from a story:
    Characters: {characters_str}
    Settings: {environments_str}
    Original story: {extracted_data["summary"]}
    
    Using the information provided, craft a compelling retelling of the story that is from a perspective other than the narrator's. Choose one of the characters or elements in the story to narrate from their point of view.
    
    First, state which character or perspective you've chosen, then write the retelling."""
    
    # Call Ollama API running locally with Mistral
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "mistral:latest",
                "prompt": prompt_template,
                "stream": False
            },
            timeout=120  # 2-minute timeout
        )
        
        # Check if request was successful
        if response.status_code == 200:
            result = response.json()
            retelling = result.get("response", "Error: No response generated")
        else:
            retelling = f"Error: Received status code {response.status_code} from Ollama API"
            print(f"API Error: {response.text}")
    
    except requests.exceptions.RequestException as e:
        retelling = f"Error connecting to Ollama: {str(e)}"
        print(f"Connection error: {str(e)}")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate unique filename with timestamp
    timestamp = time.strftime('%m-%d-%H-%M-%S')
    filename = f"{output_dir}/retell_{extracted_data['name']}_{timestamp}.csv"
    
    # Save the retelling to CSV
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["storyNum", "name", "source", "retelling"])
        writer.writeheader()
        writer.writerow({
            "storyNum": extracted_data["storyNum"],
            "name": extracted_data["name"],
            "source": extracted_data["source"],
            "retelling": retelling
        })
    
    print(f"Retelling saved to {filename}")
    return filename

def batch_generate_retellings(extracted_data_file:str, output_dir="output"):
    """Process all stories from an extracted data file"""
    
    # Load the extracted data
    with open(extracted_data_file, 'r') as f:
        extracted_data_list = json.load(f)
    
    # Process each story
    for idx, story_data in enumerate(extracted_data_list):
        print(f"Generating retelling {idx+1}/{len(extracted_data_list)} for '{story_data['name']}'")
        
        try:
            filename = generate_retelling(story_data, output_dir)
            print(f"Successfully generated retelling: {filename}")
        except Exception as e:
            print(f"Error generating retelling for story {story_data['name']}: {str(e)}")
    
    print(f"All retellings complete. Results saved to {output_dir}/")


In [ ]:
batch_generate_retellings("./extracted_20250504_032008.json")

In [14]:
# Cell 5: Structured Evaluation Functions
import requests
import json
import os
import glob
import pandas as pd
import numpy as np

def evaluate_retelling(original, retelling):
    structured_prompt = f"""Evaluate this story retelling using these EXACT criteria:

Character Selection Valence (1-5):
1: Secondary character or prominent non-narrator
5: Highly original (inanimate object/passing character)

Character Selection Error (bool):
True if character not in original story

Factual Consistency (1-5):
1: One-for-one perspective shift
5: Nuanced unexplored perspectives

Factual Consistency Error (bool):
True if major plot/genre/character discrepancy

Retelling Depth (1-5):
1: Generic perspective
5: Nuanced retelling that demonstrates a thorough understanding of another character's goals, intentions, and personality

Original Story: {original[:5000]}
Retelling: {retelling[:5000]}

Respond ONLY with JSON containing these keys:
- valence (int 1-5)
- selection_error (bool)
- consistency (int 1-5)
- fact_error (bool)
- retelling_depth (int 1-5)
- reasoning (object with keys: valence, consistency, retelling_depth)"""

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3:8b",
                "prompt": structured_prompt,
                "format": "json",
                "stream": False
            },
            timeout=300
        )
        
        if response.status_code == 200:
            try:
                result = json.loads(response.json()["response"])
                # Convert numpy types to native Python types
                return {
                    "valence": int(result["valence"]),
                    "selection_error": bool(result["selection_error"]),
                    "consistency": int(result["consistency"]),
                    "fact_error": bool(result["fact_error"]),
                    "retelling_depth": int(result["retelling_depth"]),
                    "reasoning": {
                        "valence": str(result["reasoning"]["valence"]),
                        "consistency": str(result["reasoning"]["consistency"]),
                        "retelling_depth": str(result["reasoning"]["retelling_depth"])
                    }
                }
            except (KeyError, json.JSONDecodeError) as e:
                return {"error": f"Invalid response format: {str(e)}"}
        else:
            return {"error": f"API error: {response.status_code}"}
    
    except Exception as e:
        return {"error": str(e)}

def batch_evaluate(output_dir):
    timestamp = pd.Timestamp.now().strftime("%m-%d-%H-%M-%S")
    eval_file = os.path.join(output_dir, f"eval_results_{timestamp}.jsonl")
    
    for file in glob.glob(os.path.join(output_dir, "retell_*.csv")):
        try:
            # Read CSV with dtype specification
            df = pd.read_csv(file, dtype={
                'storyNum': 'int32',
                'name': 'str',
                'source': 'str',
                'retelling': 'str'
            })
            
            # Convert pandas/numpy types to native Python types
            record = {
                "storyNum": int(df["storyNum"].iloc[0].item()),  # Convert int64 to int
                "name": str(df["name"].iloc[0]),
                "source": str(df["source"].iloc[0]),
                "retelling": str(df["retelling"].iloc[0])
            }
            
            print(f"Evaluating {record['name']}...")
            evaluation = evaluate_retelling(record["source"], record["retelling"])
            
            # Merge results with explicit type conversion
            combined = {
                **record,
                **{k: v.item() if isinstance(v, np.generic) else v 
                   for k, v in evaluation.items()}
            }
            
            # Save immediately with UTF-8 encoding
            with open(eval_file, 'a', encoding='utf-8') as f:
                json.dump(combined, f, ensure_ascii=False)
                f.write('\n')
                
        except Exception as e:
            print(f"Error processing {file}: {str(e)}")
            error_entry = {
                "file": file,
                "error": str(e),
                "timestamp": pd.Timestamp.now().isoformat()
            }
            with open(eval_file, 'a', encoding='utf-8') as f:
                json.dump(error_entry, f, ensure_ascii=False)
                f.write('\n')

    print(f"Evaluation complete. Results saved to {eval_file}")
    return eval_file


In [15]:
batch_evaluate("./output")

Evaluating DesertRevenge...
Evaluating GodsBetrayalAndRedemption...
Evaluating HarlemHold...
Evaluating DevinsCursedCoins...
Evaluating FlorentineEscape...
Evaluating AmnesiacCallCenterDevil...
Evaluating ThirtyYearsThirtyLives...
Evaluating RedHairedNord...
Evaluating FinalBirthTunnel...
Evaluating ericMayaGhost...
Evaluating NebulonTerrierIncident...
Evaluating gelliusBathDeath...
Evaluating ShipwreckedGenieWish...
Evaluating woodsBodyMystery...
Evaluating CokeEscape...
Evaluating BlackHatWhiteHatRebirth...
Evaluating LostLifetimeJourney...
Evaluating HiddenDifference...
Evaluating FallenSoldier...
Evaluating OrphanageAnomaly...
Evaluating SuicideHotlineCall...
Evaluating OceanSafeRepository...
Evaluating SuddenDarkness...
Evaluating JerrysVirtualVacation...
Evaluating GameChildrensDiscovery...
Evaluating EarthCommandPrompt...
Evaluating SilentFogEncounter...
Evaluating LicenseAndRegistrationPlease...
Evaluating ForgottenCelebration...
Evaluating GoneBabyBoy...
Evaluating GrandpaRedD

'./output/eval_results_05-05-03-25-12.jsonl'